# Bikol NMT Capstone: English, Virac (Southern Bikol) and Pandan (Northern Bikol)
NLLB-200 (600M) + LoRA, a translation-memory (RAG) layer, FastAPI + Bootstrap UI, and BLEU / chrF++ / TER evaluation.

**Directions trained:** English to Virac, English to Pandan, Virac to Pandan (edit `DIRECTIONS` in `config.py` to add more).

**Before you start:** Runtime > Change runtime type > **T4 GPU**, and copy `Team2_Bikol_Translation_Dataset_1_5k_.csv` into your Google Drive folder `MyDrive/bikol_nmt/data/`. Nothing else needs uploading: every script is written by the notebook. Run the cells top to bottom; each step is tagged with the PDF phase/task it satisfies.

## Step 0: Setup (GPU, packages, Drive)

In [ ]:
!nvidia-smi
!pip install -q -U transformers datasets peft accelerate evaluate sacrebleu sentencepiece scikit-learn fastapi "uvicorn[standard]" huggingface_hub
# Colab ships an old torchao that newer peft refuses to load; we do not use it, so remove it
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT = "/content/drive/MyDrive/bikol_nmt"
os.makedirs(f"{PROJECT}/data", exist_ok=True)
os.chdir(PROJECT)
print(os.listdir(PROJECT), os.listdir(PROJECT + "/data"))

## Step 1: Shared config
Everything tunable lives here. **Check `FALLBACK`**: it assumes `mai ko isi` for Pandan (North) and `dai ko isi` for Virac (South), chosen by the *target* language of the translation. Swap them if your instructor meant the opposite.

In [ ]:
%%writefile /content/drive/MyDrive/bikol_nmt/config.py
PROJECT = "/content/drive/MyDrive/bikol_nmt"
DATA_DIR = PROJECT + "/data/processed"
ADAPTER_DIR = PROJECT + "/adapter"
TM_PATH = PROJECT + "/memory.csv"
TM_TRAIN_ONLY_PATH = PROJECT + "/memory_train_only.csv"
BASE_MODEL = "facebook/nllb-200-distilled-600M"
MAX_LEN = 128

# ---- Translation directions to train (source, target) ----
DIRECTIONS = [
    ("eng_Latn", "bcl_South"),    # English -> Virac (Southern Bikol)
    ("eng_Latn", "bcl_North"),    # English -> Pandan (Northern Bikol)
    ("bcl_South", "bcl_North"),   # Virac -> Pandan
]
# To also train Bikol -> English (the PDF's API example uses bcl_North -> eng_Latn), add:
#   ("bcl_North", "eng_Latn"), ("bcl_South", "eng_Latn")
# then re-run Step 2 (data) and Step 4 (training).

LANG_NAME = {"eng_Latn": "English", "bcl_South": "Virac (Southern Bikol)", "bcl_North": "Pandan (Northern Bikol)"}
TGT_WORD = {"eng_Latn": "english", "bcl_South": "virac", "bcl_North": "pandan"}

# NLLB-200 has no Bikol language tag, so both Bikol varieties borrow the Tagalog tag.
# The real target (virac / pandan / english) is told to the model with a text prefix.
NLLB_TAG = {"eng_Latn": "eng_Latn", "bcl_South": "tgl_Latn", "bcl_North": "tgl_Latn"}

def make_input(text, tgt):
    return ">>" + TGT_WORD[tgt] + "<< " + text

# Fallback phrase. Chosen by the TARGET language (the dialect the system would answer in).
FALLBACK = {"bcl_North": "mai ko isi", "bcl_South": "dai ko isi"}

# LoRA / training (PDF Tasks 2.2 and 2.3)
LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["q_proj", "v_proj"])
EPOCHS = 20
LR = 3e-4

# "Do I know this?" thresholds (tuned in Step 8)
EXACT_SIM = 0.97       # >= this: answer straight from translation memory
FUZZY_SIM = 0.60       # below this the sentence is not close to anything known
MIN_COVERAGE = 0.90    # tuned in Step 8b: catches ~91% of off-topic test sentences (in-domain false alarms ~18%)
MIN_CONF = 0.30        # model confidence

## PHASE 1: Data (Tasks 1.1 to 1.4)
Task 1.1 (corpus mining) is done: you have the dataset. Step 2 writes the cleaning script and runs it: clean text, remove duplicates, split 70/15/15 by row with zero leakage, build one pair per direction, and compute vocabulary statistics.

In [ ]:
%%writefile /content/drive/MyDrive/bikol_nmt/prep_data.py
import argparse, html, json, re, unicodedata
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from config import DIRECTIONS

# Raw file columns: Number, English, Virac (= Southern Bikol), Pandan (= Northern Bikol).
# (The raw CSV also has ~16,000 empty columns; usecols below ignores them.)
RAW_COL = {"eng_Latn": "English", "bcl_South": "Virac", "bcl_North": "Pandan"}
SLASH_KEEP_FIRST = True   # "isisila/lulutuon" -> "isisila" (Bikol columns only)


def clean(text, bikol):
    t = html.unescape(str(text))
    t = re.sub(r"<[^>]+>", " ", t)                       # strip HTML
    t = unicodedata.normalize("NFC", t)                  # standardise diacritics
    for a, b in {"\u2019": "'", "\u2018": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-"}.items():
        t = t.replace(a, b)
    if bikol and SLASH_KEEP_FIRST:
        t = re.sub(r"(\S+?)/\S+", r"\1", t)              # keep first spelling variant
    t = re.sub(r"[^\w\s.,!?;:'\"()\-/]", " ", t)         # stray special characters
    return re.sub(r"\s+", " ", t).strip()                # also fixes \n and double spaces


def norm(t):
    return re.sub(r"[^\w\s]", "", str(t).lower()).strip()


def to_pairs(part):
    frames = []
    for s, t in DIRECTIONS:
        frames.append(pd.DataFrame({
            "source_text": part[RAW_COL[s]].values, "target_text": part[RAW_COL[t]].values,
            "source_lang": s, "target_lang": t, "row_id": part["Number"].values}))
    return pd.concat(frames, ignore_index=True)


def main(a):
    raw = pd.read_csv(a.input, usecols=["Number", "English", "Virac", "Pandan"]).dropna()
    log = {"raw_rows": len(raw)}
    raw["English"] = raw["English"].map(lambda x: clean(x, False))
    raw["Virac"] = raw["Virac"].map(lambda x: clean(x, True))
    raw["Pandan"] = raw["Pandan"].map(lambda x: clean(x, True))
    raw = raw[(raw[["English", "Virac", "Pandan"]] != "").all(axis=1)]
    raw = raw.assign(_k=raw["English"].map(norm)).drop_duplicates("_k").drop(columns="_k").reset_index(drop=True)
    log["rows_after_cleaning"] = len(raw)

    # Task 1.3: split BY ROW (one row = the same sentence in 3 languages). Every direction of a
    # row therefore lands in the same split, so a sentence can never be seen in training in one
    # direction and tested in another.
    train_r, rest = train_test_split(raw, test_size=0.30, random_state=42)
    val_r, test_r = train_test_split(rest, test_size=0.50, random_state=42)
    train, val, test = to_pairs(train_r), to_pairs(val_r), to_pairs(test_r)

    # Extra leakage guard: drop val/test pairs whose source text (in that source language)
    # also occurs in train (a few Bikol sentences are repeated with different English).
    seen = set(zip(train.source_lang, train.source_text.map(norm)))
    def guard(df, name):
        keep = [(l, norm(s)) not in seen for l, s in zip(df.source_lang, df.source_text)]
        log[f"{name}_removed_for_leakage"] = int(len(df) - sum(keep))
        return df[keep]
    val, test = guard(val, "val"), guard(test, "test")
    train = train.drop_duplicates(subset=["source_lang", "target_lang", "source_text", "target_text"])
    val = val.drop_duplicates(subset=["source_lang", "target_lang", "source_text"])
    test = test.drop_duplicates(subset=["source_lang", "target_lang", "source_text"])
    for part in (val, test):
        assert not set(zip(part.source_lang, part.source_text.map(norm))) & seen

    out = Path(a.outdir); out.mkdir(parents=True, exist_ok=True)
    for name, part in [("train", train), ("validation", val), ("test", test)]:
        part.sample(frac=1, random_state=0).to_csv(out / f"{name}.csv", index=False)
        log[f"{name}_pairs"] = len(part)
        log[f"{name}_by_direction"] = {f"{s}->{t}": int(n) for (s, t), n in part.groupby(["source_lang", "target_lang"]).size().items()}

    # Task 1.4: baseline vocabulary analysis
    stats = {"log": log}
    cols = {"English": raw["English"], "Virac (South)": raw["Virac"], "Pandan (North)": raw["Pandan"]}
    vocab = {}
    for name, ser in cols.items():
        toks = [w for s in ser for w in norm(s).split()]
        lens = ser.map(lambda s: len(norm(s).split()))
        vocab[name] = set(toks)
        stats[name] = {"sentences": int(len(ser)), "tokens": len(toks), "types": len(set(toks)),
                       "TTR": round(len(set(toks)) / len(toks), 4), "len_mean": round(float(lens.mean()), 2),
                       "len_median": float(lens.median()), "len_max": int(lens.max())}
    n, s = vocab["Pandan (North)"], vocab["Virac (South)"]
    stats["Virac_Pandan_vocab_jaccard"] = round(len(n & s) / len(n | s), 4)
    stats["rows_where_Virac_equals_Pandan"] = int((raw["Pandan"].map(norm) == raw["Virac"].map(norm)).sum())
    (out / "stats.json").write_text(json.dumps(stats, indent=2))
    print(json.dumps(stats, indent=2))

    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    for name, ser in cols.items():
        plt.hist(ser.map(lambda x: len(x.split())), bins=range(1, 25), alpha=.5, label=name)
    plt.xlabel("words per sentence"); plt.ylabel("sentences"); plt.legend()
    plt.savefig(out / "length_distribution.png", dpi=150)


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--input", required=True)
    p.add_argument("--outdir", default="data/processed")
    main(p.parse_args())

In [ ]:
!python prep_data.py --input "data/Team2_Bikol_Translation_Dataset_1_5k_.csv" --outdir data/processed

In [ ]:
import pandas as pd
print(pd.read_csv("data/processed/train.csv").head(8).to_string())
for sp in ["train", "validation", "test"]:
    d = pd.read_csv(f"data/processed/{sp}.csv")
    print(sp, len(d), d.groupby(["source_lang", "target_lang"]).size().to_dict())
from IPython.display import Image; Image("data/processed/length_distribution.png")

## PHASE 2: LoRA fine-tuning (Tasks 2.1 to 2.4)
`train.py` is a required deliverable, so it is written as a real script. It resumes automatically from the last checkpoint if Colab disconnects (checkpoints are saved to Drive). One model learns all three directions: the target is selected by a short text prefix such as `>>virac<<`.

In [ ]:
%%writefile /content/drive/MyDrive/bikol_nmt/train.py
import glob, json, math, torch, pandas as pd
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback)
from peft import LoraConfig, get_peft_model, TaskType
from config import *

tok = AutoTokenizer.from_pretrained(BASE_MODEL)

def load(split):
    return Dataset.from_pandas(pd.read_csv(f"{DATA_DIR}/{split}.csv"), preserve_index=False)

def encode(ex):
    tok.src_lang = NLLB_TAG[ex["source_lang"]]
    tok.tgt_lang = NLLB_TAG[ex["target_lang"]]
    return tok(make_input(ex["source_text"], ex["target_lang"]), text_target=ex["target_text"],
               max_length=MAX_LEN, truncation=True)

train_ds, val_ds = load("train"), load("validation")
train_ds = train_ds.map(encode, remove_columns=train_ds.column_names)
val_ds = val_ds.map(encode, remove_columns=val_ds.column_names)

# Task 2.1 base model, Task 2.2 LoRA on q_proj / v_proj
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
model = get_peft_model(model, LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, **LORA))
model.print_trainable_parameters()

# Task 2.3 AdamW + linear decay with warmup, loss monitored every epoch
steps_per_epoch = math.ceil(len(train_ds) / (16 * 2))     # batch 16 x grad-accum 2
WARMUP = int(0.05 * steps_per_epoch * EPOCHS)              # 5% warmup (works on old and new transformers)
args = Seq2SeqTrainingArguments(
    output_dir=f"{PROJECT}/checkpoints",
    num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=16, per_device_eval_batch_size=16, gradient_accumulation_steps=2,
    warmup_steps=WARMUP, lr_scheduler_type="linear", optim="adamw_torch", weight_decay=0.01,
    fp16=True,                                   # T4 has no bf16
    eval_strategy="epoch", save_strategy="epoch", logging_strategy="epoch",
    save_total_limit=2, load_best_model_at_end=True,
    metric_for_best_model="eval_loss", greater_is_better=False,
    report_to="none", seed=42,
)
trainer = Seq2SeqTrainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tok, model=model),
    processing_class=tok,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)
resume = len(glob.glob(f"{PROJECT}/checkpoints/checkpoint-*")) > 0   # survive Colab disconnects
trainer.train(resume_from_checkpoint=resume)

# Task 2.4 save only the LoRA adapter (small)
model.save_pretrained(ADAPTER_DIR)
json.dump(trainer.state.log_history, open(f"{PROJECT}/train_log.json", "w"))
print("Adapter saved to", ADAPTER_DIR)

In [ ]:
!python train.py

Training-vs-validation loss per epoch (PDF Task 2.3). Put this plot in your report.

In [ ]:
import json, matplotlib.pyplot as plt
log = json.load(open("train_log.json"))
tr = [(l["epoch"], l["loss"]) for l in log if "loss" in l]
ev = [(l["epoch"], l["eval_loss"]) for l in log if "eval_loss" in l]
plt.plot(*zip(*tr), label="train loss"); plt.plot(*zip(*ev), label="validation loss")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=.3); plt.savefig("loss_curve.png", dpi=150); plt.show()
!ls -lh adapter

**Upload the adapter to the Hugging Face Hub** (required deliverable). Create a token at huggingface.co/settings/tokens (write access) and replace `YOUR_USERNAME`.

In [ ]:
from huggingface_hub import login, HfApi
login()
api = HfApi()
repo = "YOUR_USERNAME/bikol-nllb-lora"
api.create_repo(repo, exist_ok=True)
api.upload_folder(folder_path="adapter", repo_id=repo)

## Step 7: Inference engine + translation memory (RAG) + "mai ko isi / dai ko isi"
`engine.py` is shared by the notebook and the API. For every request:
1. **Translation memory lookup** (character n-gram TF-IDF over all known source sentences for that direction). Near-exact match: return the stored translation.
2. Otherwise the **LoRA model** translates and returns a confidence score.
3. If the sentence is **not close to anything known** *and* (too many unknown words *or* low confidence), return the fallback phrase instead of a guess.

New data goes into the memory (English + Virac + Pandan triplets) with no retraining.

In [ ]:
%%writefile /content/drive/MyDrive/bikol_nmt/engine.py
import re, time, torch, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
from config import *

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COLS = ["source_text", "target_text", "source_lang", "target_lang"]
TRIPLET_COL = {"eng_Latn": "english", "bcl_South": "virac", "bcl_North": "pandan"}

def norm(s):
    return re.sub(r"[^\w\s]", "", str(s).lower()).strip()

def expand_triplets(df):
    # df has columns english, virac, pandan -> one row per trained direction
    parts = [pd.DataFrame({"source_text": df[TRIPLET_COL[s]].astype(str).str.strip().values,
                           "target_text": df[TRIPLET_COL[t]].astype(str).str.strip().values,
                           "source_lang": s, "target_lang": t}) for s, t in DIRECTIONS]
    return pd.concat(parts, ignore_index=True)

def load_model(adapter=True, merge=True):
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL, torch_dtype=dtype)
    if adapter:
        model = PeftModel.from_pretrained(model, ADAPTER_DIR)
        if merge:
            model = model.merge_and_unload()   # faster inference, no adapter overhead
    return tok, model.to(DEVICE).eval()

@torch.no_grad()
def translate_batch(tok, model, texts, src, tgt, prefix=True, beams=4, max_new=128):
    # one batch = one direction (src -> tgt). beams must be >= 2 (confidence uses beam scores).
    tok.src_lang = NLLB_TAG[src]
    if prefix:
        texts = [make_input(t, tgt) for t in texts]
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(DEVICE)
    out = model.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(NLLB_TAG[tgt]),
                         num_beams=beams, max_length=max_new,
                         output_scores=True, return_dict_in_generate=True)
    hyps = tok.batch_decode(out.sequences, skip_special_tokens=True)
    conf = torch.exp(out.sequences_scores.float()).tolist()   # length-normalised probability
    return hyps, conf

class TranslationMemory:
    def __init__(self):
        self.df = pd.DataFrame(columns=COLS)
        self.vec, self.X, self.vocab = None, None, {}

    def add(self, rows):
        self.df = pd.concat([self.df, rows[COLS]], ignore_index=True).drop_duplicates().reset_index(drop=True)
        keys = self.df["source_text"].map(norm)
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5))
        self.X = self.vec.fit_transform(keys)          # rows are L2-normalised -> dot product = cosine
        self.vocab = {lang: {w for s in keys[self.df["source_lang"] == lang] for w in s.split()}
                      for lang in self.df["source_lang"].unique()}
        return self

    def lookup(self, text, src, tgt):
        mask = ((self.df["source_lang"] == src) & (self.df["target_lang"] == tgt)).values
        if not mask.any():
            return 0.0, None
        sims = (self.X[mask] @ self.vec.transform([norm(text)]).T).toarray().ravel()
        i = int(sims.argmax())
        return float(sims[i]), self.df[mask].iloc[i]

    def coverage(self, text, src):
        ws = norm(text).split()
        v = self.vocab.get(src, set())
        return sum(w in v for w in ws) / max(1, len(ws))

    def save(self, path):
        self.df.to_csv(path, index=False)

    @classmethod
    def load(cls, path):
        return cls().add(pd.read_csv(path))

def should_fallback(sim, cov, conf):
    return sim < FUZZY_SIM and (cov < MIN_COVERAGE or conf < MIN_CONF)

class BikolTranslator:
    def __init__(self, tm_path=TM_PATH):
        self.tok, self.model = load_model()
        self.tm = TranslationMemory.load(tm_path)

    def translate(self, text, source_lang, target_lang):
        t0 = time.perf_counter()
        sim, row = self.tm.lookup(text, source_lang, target_lang)
        if row is not None and sim >= EXACT_SIM:
            out, conf = row["target_text"], sim
        else:
            hyps, confs = translate_batch(self.tok, self.model, [text], source_lang, target_lang)
            out, conf = hyps[0], confs[0]
            if should_fallback(sim, self.tm.coverage(text, source_lang), conf):
                out = FALLBACK.get(target_lang) or FALLBACK[source_lang]
        ms = int((time.perf_counter() - t0) * 1000)
        return {"status": "success", "translated_text": out,
                "confidence_score": round(conf, 2), "inference_latency_ms": ms}

### Build the translation memories
- `memory_train_only.csv`: used for honest evaluation and threshold tuning (the test set is **not** in it, otherwise you'd get fake exact matches).
- `memory.csv`: everything; used by the live app.

In [ ]:
import pandas as pd
from config import *
from engine import *
tr = pd.read_csv(f"{DATA_DIR}/train.csv"); va = pd.read_csv(f"{DATA_DIR}/validation.csv"); te = pd.read_csv(f"{DATA_DIR}/test.csv")
TranslationMemory().add(tr).save(TM_TRAIN_ONLY_PATH)
TranslationMemory().add(pd.concat([tr, va, te])).save(TM_PATH)
print("memories built")

## Step 8: Tune the 'mai ko isi / dai ko isi' thresholds
This shows the three signals for **in-domain** test sentences vs **out-of-domain** text. Your data is cooking only, so anything else should trigger the fallback. **Replace the `OOD_BIKOL` examples with real Virac sentences about other topics** (school, weather, family...): those are what an instructor is most likely to type. You want the in-domain fallback rate near 0 and the out-of-domain rate near 1. If not, edit the thresholds at the bottom of `config.py`, run the reload line, and re-run this cell.

In [ ]:
import importlib, config, engine
importlib.reload(config); importlib.reload(engine)
from engine import *
tok, model = load_model()
tm = TranslationMemory.load(TM_TRAIN_ONLY_PATH)
test = pd.read_csv(f"{DATA_DIR}/test.csv").sample(frac=1, random_state=0).head(150)

OOD_EN = ["What is the capital of France?", "Please explain how photosynthesis works",
          "The quick brown fox jumps over the lazy dog", "Write a Python function to sort a list",
          "Where is the nearest hospital?", "My phone battery is almost empty"]
OOD_BIKOL = ["Kumusta ka na, saan ka pupunta?",            # <- replace with real Virac sentences
             "Maghapon ang ulan kanina sa eskwelahan"]     #    about NON-cooking topics

rows = [("in-domain", r.source_lang, r.target_lang, r.source_text) for r in test.itertuples()]
rows += [("out-of-domain", "eng_Latn", t, s) for s in OOD_EN for t in ("bcl_South", "bcl_North")]
rows += [("out-of-domain", "bcl_South", "bcl_North", s) for s in OOD_BIKOL]

def signals(src, tgt, text):
    sim, _ = tm.lookup(text, src, tgt)
    _, c = translate_batch(tok, model, [text], src, tgt)
    return sim, tm.coverage(text, src), c[0]

sig = pd.DataFrame([(g,) + signals(s, t, x) for g, s, t, x in rows], columns=["group", "sim", "coverage", "conf"])
sig["fallback"] = [should_fallback(a, b, c) for a, b, c in zip(sig.sim, sig.coverage, sig.conf)]
print(sig.groupby("group")[["sim", "coverage", "conf", "fallback"]].mean().round(3))

### Step 8b: Which off-topic sentences slipped through, and what threshold fixes it?
Run this right after Step 8 (it reuses its results). It lists the off-topic sentences that did NOT trigger the fallback and tests several `MIN_COVERAGE` values. Pick the value where in-domain stays near 0 and out-of-domain is as high as possible, then set it in `config.py`.

In [ ]:
sig["text"] = [r[3] for r in rows]
missed = sig[(sig.group == "out-of-domain") & (~sig.fallback)]
print("Off-topic sentences that were NOT caught:"); print(missed[["text", "sim", "coverage", "conf"]].round(2).to_string())
print("\nIn-domain coverage distribution:"); print(sig[sig.group == "in-domain"].coverage.describe(percentiles=[.05, .1, .25]).round(2))
out = []
for cov in [0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]:
    fb = [(a < FUZZY_SIM) and (b < cov or c < MIN_CONF) for a, b, c in zip(sig.sim, sig.coverage, sig.conf)]
    out.append((cov, pd.Series(fb)[sig.group.values == "in-domain"].mean(), pd.Series(fb)[sig.group.values == "out-of-domain"].mean()))
print("\nMIN_COVERAGE sweep"); print(pd.DataFrame(out, columns=["MIN_COVERAGE", "in-domain fallback", "out-of-domain fallback"]).round(3).to_string(index=False))

## PHASE 3: REST API + web UI (Tasks 3.1 and 3.2)
`POST /api/v1/translate` uses the PDF's payload format (`source_text`, `source_lang`, `target_lang` in; `status`, `translated_text`, `confidence_score`, `inference_latency_ms` out), with language codes `eng_Latn`, `bcl_South` (Virac) and `bcl_North` (Pandan). Extras: `GET /api/v1/directions`, `POST /api/v1/feedback` (the 'Report Inaccurate Translation' button, saved to SQLite) and `POST /api/v1/memory` (add new sentence triplets with no retraining).

In [ ]:
%%writefile /content/drive/MyDrive/bikol_nmt/main.py
import sqlite3
import pandas as pd
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel
from config import *
from engine import BikolTranslator, expand_triplets

app = FastAPI(title="Bikol NMT")
tr = BikolTranslator()
db = sqlite3.connect(f"{PROJECT}/feedback.db", check_same_thread=False)
db.execute("""CREATE TABLE IF NOT EXISTS feedback (id INTEGER PRIMARY KEY AUTOINCREMENT,
    source_text TEXT, source_lang TEXT, target_lang TEXT, model_output TEXT,
    suggested_text TEXT, created_at TEXT DEFAULT CURRENT_TIMESTAMP)""")

class TranslateIn(BaseModel):
    source_text: str
    source_lang: str
    target_lang: str

class FeedbackIn(TranslateIn):
    model_output: str
    suggested_text: str | None = None

class Triplet(BaseModel):
    english: str
    virac: str
    pandan: str

@app.get("/")
def index():
    return FileResponse(f"{PROJECT}/index.html")

@app.get("/api/v1/directions")
def directions():
    return {"directions": [list(d) for d in DIRECTIONS], "names": LANG_NAME}

@app.post("/api/v1/translate")
def translate(p: TranslateIn):
    if (p.source_lang, p.target_lang) not in DIRECTIONS or not p.source_text.strip():
        raise HTTPException(status_code=400, detail="direction not supported or empty source_text")
    return tr.translate(p.source_text.strip(), p.source_lang, p.target_lang)

@app.post("/api/v1/feedback")
def feedback(f: FeedbackIn):
    db.execute("INSERT INTO feedback (source_text, source_lang, target_lang, model_output, suggested_text) VALUES (?,?,?,?,?)",
               (f.source_text, f.source_lang, f.target_lang, f.model_output, f.suggested_text))
    db.commit()
    return {"status": "success"}

@app.post("/api/v1/memory")
def add_memory(rows: list[Triplet]):
    # RAG: add new sentences (English + Virac + Pandan) with NO retraining
    tr.tm.add(expand_triplets(pd.DataFrame([r.model_dump() for r in rows])))
    tr.tm.save(TM_PATH)
    return {"status": "success", "memory_size": len(tr.tm.df)}

In [ ]:
%%writefile /content/drive/MyDrive/bikol_nmt/index.html
<!doctype html>
<html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Bikol Translator</title>
<link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css" rel="stylesheet"></head>
<body class="bg-light"><div class="container py-4">
<h3 class="mb-3">Bikol Translator <small class="text-muted fs-6">Virac / Pandan / English</small></h3>
<div class="row g-3">
  <div class="col-md-6">
    <select id="src" class="form-select mb-2"></select>
    <textarea id="inp" class="form-control" rows="7" placeholder="Type text..."></textarea></div>
  <div class="col-md-6">
    <select id="tgt" class="form-select mb-2"></select>
    <textarea id="out" class="form-control" rows="7" readonly></textarea></div>
</div>
<div class="mt-3 d-flex gap-2 align-items-center">
  <button id="copy" class="btn btn-outline-secondary">Copy</button>
  <button id="report" class="btn btn-outline-danger">Report Inaccurate Translation</button>
  <span id="meta" class="text-muted small ms-2"></span>
</div></div>
<script>
const $ = id => document.getElementById(id); let timer, DIRS = [], NAMES = {};
function fillTargets() {
  const ts = DIRS.filter(d => d[0] === $('src').value).map(d => d[1]);
  $('tgt').innerHTML = ts.map(t => '<option value="' + t + '">' + NAMES[t] + '</option>').join('');
}
async function init() {
  const d = await (await fetch('api/v1/directions')).json(); DIRS = d.directions; NAMES = d.names;
  const srcs = [...new Set(DIRS.map(x => x[0]))];
  $('src').innerHTML = srcs.map(s => '<option value="' + s + '">' + NAMES[s] + '</option>').join('');
  fillTargets();
}
async function translate() {
  const text = $('inp').value.trim();
  if (!text) { $('out').value = ''; $('meta').textContent = ''; return; }
  const r = await fetch('api/v1/translate', {method: 'POST', headers: {'Content-Type': 'application/json'},
    body: JSON.stringify({source_text: text, source_lang: $('src').value, target_lang: $('tgt').value})});
  const d = await r.json();
  $('out').value = d.translated_text || d.detail || 'Error';
  $('meta').textContent = d.status === 'success' ? 'confidence ' + d.confidence_score + ' | ' + d.inference_latency_ms + ' ms' : '';
}
$('src').addEventListener('change', () => { fillTargets(); translate(); });
['inp', 'tgt'].forEach(id => $(id).addEventListener('input', () => { clearTimeout(timer); timer = setTimeout(translate, 500); }));
$('copy').onclick = () => { $('out').select(); document.execCommand('copy'); };
$('report').onclick = async () => {
  if (!$('out').value) return;
  const s = prompt('Correct translation (optional):'); if (s === null) return;
  await fetch('api/v1/feedback', {method: 'POST', headers: {'Content-Type': 'application/json'},
    body: JSON.stringify({source_text: $('inp').value, source_lang: $('src').value, target_lang: $('tgt').value,
                          model_output: $('out').value, suggested_text: s || null})});
  alert('Thanks! Saved for retraining.');
};
init();
</script></body></html>

In [ ]:
import sys, threading, uvicorn
sys.path.insert(0, PROJECT)
import main
threading.Thread(target=lambda: uvicorn.run(main.app, host="0.0.0.0", port=8000, log_level="warning"), daemon=True).start()

from google.colab import output
output.serve_kernel_port_as_iframe(8000, height=650)

**Test the API directly** and **prove the RAG requirement (add data with no retraining)**:

In [ ]:
import requests
u = "http://localhost:8000/api/v1/"
print(requests.post(u + "translate", json={"source_text": "Slice the onions and garlic into small pieces", "source_lang": "eng_Latn", "target_lang": "bcl_North"}).json())
print(requests.post(u + "translate", json={"source_text": "What is the capital of France?", "source_lang": "eng_Latn", "target_lang": "bcl_South"}).json())

# add a brand-new sentence in all three languages, then translate it immediately: no retraining
new = [{"english": "REPLACE with a new English sentence", "virac": "REPLACE with its Virac version", "pandan": "REPLACE with its Pandan version"}]
print(requests.post(u + "memory", json=new).json())
print(requests.post(u + "translate", json={"source_text": new[0]["english"], "source_lang": "eng_Latn", "target_lang": "bcl_South"}).json())

## PHASE 4: Evaluation (Tasks 4.1 to 4.3)
This evaluates the **model only** (no memory), so the numbers reflect the fine-tuning and not lookup hits. For Virac to Pandan an extra row reports only the sentences where the two dialects differ, because identical sentences (a copy task) would inflate the score.

In [ ]:
import torch, gc, sacrebleu, pandas as pd
from config import *
from engine import *

test = pd.read_csv(f"{DATA_DIR}/test.csv")

def run(tok, model, prefix):
    hyps = [None] * len(test)
    for (s, t), g in test.groupby(["source_lang", "target_lang"]):
        idx = g["source_text"].str.len().sort_values().index.tolist()
        for i in range(0, len(idx), 32):
            b = idx[i:i + 32]
            h, _ = translate_batch(tok, model, test.loc[b, "source_text"].tolist(), s, t, prefix=prefix)
            for j, x in zip(b, h): hyps[j] = x
    return hyps

def metrics(h, r):
    return {"BLEU": sacrebleu.corpus_bleu(h, [r]).score,
            "chrF++": sacrebleu.corpus_chrf(h, [r], word_order=2).score,
            "TER": sacrebleu.corpus_ter(h, [r]).score}

def subsets():
    yield "ALL", test
    for (s, t), g in test.groupby(["source_lang", "target_lang"]):
        yield f"{s} -> {t}", g
        if (s, t) == ("bcl_South", "bcl_North"):
            yield f"{s} -> {t} (non-identical only)", g[g.source_text.map(norm) != g.target_text.map(norm)]

# Task 4.2: zero-shot baseline (un-tuned NLLB, Tagalog tag for Bikol, no prefix) vs LoRA
tok, base = load_model(adapter=False)
test["hyp_zeroshot"] = run(tok, base, prefix=False)
del base; gc.collect(); torch.cuda.empty_cache()

tok, tuned = load_model(adapter=True)
test["hyp_lora"] = run(tok, tuned, prefix=True)

rows = []
for name, col in [("Zero-shot NLLB-600M", "hyp_zeroshot"), ("Fine-tuned LoRA", "hyp_lora")]:
    for sub, g in subsets():
        rows.append({"model": name, "subset": sub, "n": len(g), **metrics(g[col].tolist(), g.target_text.tolist())})
results = pd.DataFrame(rows).round(2)
results.to_csv("results_metrics.csv", index=False); test.to_csv("test_predictions.csv", index=False)
results

### Latency table (inference time per token length; target < 300 ms)
If the mean is above 300 ms, lower `beams` in `translate_batch` to 2 (it must stay >= 2).

In [ ]:
import time
sample = test.sample(min(100, len(test)), random_state=1)
translate_batch(tok, tuned, ["warm up"], "eng_Latn", "bcl_South")   # warm-up
rows = []
for r in sample.itertuples():
    n = len(tok(r.source_text).input_ids)
    torch.cuda.synchronize(); t = time.perf_counter()
    translate_batch(tok, tuned, [r.source_text], r.source_lang, r.target_lang)
    torch.cuda.synchronize(); rows.append((n, (time.perf_counter() - t) * 1000))
lat = pd.DataFrame(rows, columns=["tokens", "ms"])
lat["token_bucket"] = pd.cut(lat.tokens, [0, 10, 20, 30, 50, 500])
latency_table = lat.groupby("token_bucket", observed=True).ms.agg(["count", "mean", "median", "max"]).round(1)
latency_table.to_csv("results_latency.csv"); latency_table

### Task 4.3: error analysis sheet
Exports the 50 worst translations (lowest sentence-level chrF). Open `error_analysis.csv` and fill the `error_category` column using the PDF's categories: **OOV**, **Morphological/Prefix**, **Word order**. `oov_words` lists source words never seen in training (for that source language) to speed up the OOV judgement.

In [ ]:
trn = pd.read_csv(f"{DATA_DIR}/train.csv")
train_vocab = {l: {w for s in g.source_text.map(norm) for w in s.split()} for l, g in trn.groupby("source_lang")}
test["sent_chrf"] = [sacrebleu.sentence_chrf(h, [r], word_order=2).score for h, r in zip(test.hyp_lora, test.target_text)]
worst = test.sort_values("sent_chrf").head(50).copy()
worst["oov_words"] = [" ".join(w for w in norm(s).split() if w not in train_vocab[l]) for s, l in zip(worst.source_text, worst.source_lang)]
worst["error_category"] = ""
worst[["source_lang", "target_lang", "source_text", "target_text", "hyp_lora", "sent_chrf", "oov_words", "error_category"]].to_csv("error_analysis.csv", index=False)
worst[["source_lang", "target_lang", "source_text", "target_text", "hyp_lora", "oov_words"]].head(10)

## Done
Deliverables now in `MyDrive/bikol_nmt/`: `data/processed/*`, `adapter/`, `main.py` + `index.html` + `engine.py` + `config.py`, `train.py`, `prep_data.py`, `results_metrics.csv`, `results_latency.csv`, `error_analysis.csv`, `loss_curve.png`. Still to write: `README.md`, and push the repo to GitHub.